# $\hat p^\star$ for the **distance route** — generated Kingman pool

> **Purpose:** the distance-route mirror of the paper's **Figure 3**
> (`fig:pstar_gen`, from `sec5_empirical/generated/eta_pool_sweep.ipynb`).
> Same two-panel-per-$\eta$ layout, same NMI metric, same $\hat p^\star$ read-off,
> same estimator for $C$ — but the operator is the **centred distance matrix**, not the
> normalized Laplacian.

## What is being measured

Theorem 2 (`thm:main-dist`) recovers the clan split from the sign pattern of
$\mathbf u_1$, the eigenvector of

$$\mathcal B = H\,\mathcal D\,H, \qquad H = I - \tfrac1m\mathbf{11}^\top,
  \qquad \mathcal D_{ij} = -\log S_{ij},\ \ \mathcal D_{ii}=0$$

at the eigenvalue of largest **absolute** value ($\mathcal B\preceq0$ for a raw distance
matrix, `lem:split-decomp`, so that is the most negative eigenvalue — not the top algebraic
one, whose eigenvector is the degenerate $\mathbf 1$ null direction).

Data is the **same $\eta$-binned Kingman pool** Figure 3 uses — dendropy Kingman
coalescent trees, JC69 sequences of length $\ell=10^4$, cached in `cache/pool_sample/`. No
re-simulation: each pool entry stores the JC *similarity* matrix $M$, and the distance is
recovered losslessly as $\mathcal D=-\log M$. So the trees, the sequences and the $\eta$
bins are shared with the similarity figure; only the operator differs.

## Two scope caveats — read before using these panels

**1. Kingman trees are outside Theorem 2's certified regime at every $\eta$.** Hypotheses
(i)–(ii) of `thm:main-dist` are verified only in the *balanced flat CBM*
(`cor:dist-balanced`), which `rem:flat-realization` notes is two stars — not a binary tree,
and certainly not a coalescent tree. `cor:dist-balanced` closes by saying so explicitly:
"at general imbalance, and on binary trees, they remain open." These panels therefore
*measure* what the estimator does on realistic trees; they do not confirm a certificate.
`DISTANCE_REVIEW.md` §7.2 and F13 both warn about presenting such a sweep without that
sentence attached.

**2. This is a layout mirror of Figure 3, NOT a head-to-head against it.** The two use
different rounding rules and different rep accounting:

| | rounding rule | reps handling |
|---|---|---|
| **Fig 3** (similarity) | $k$-means($k$=2) on $L_{\rm sym}$, applied to the **bootstrap-average** of 10 aligned sub-sampled Fiedler vectors | NMI of the averaged vector |
| **this notebook** (distance) | $\mathrm{sign}(\mathbf u_1)$ — the rule Theorem 2 actually analyses | NMI per rep, then averaged |

Averaged per-rep NMI is not the NMI of the averaged vector, and Fig 3 additionally
short-circuits $p\ge0.9999\to\text{NMI}=1$ and early-stops after two consecutive 100 %
(`src/runners/p_sweep_inner.py:169-185`) — neither of which happens here. The distance
curve is therefore expected to sit *below* Fig 3's at matched $p$ for reasons that are
partly protocol, not operator. Reading the two as a comparison of routes is the mistake
that already got two numbers struck from the manuscript (`open-items/13-B1.md` [B1/07],
`DISTANCE_REVIEW.md` §6.2(8)).

The **synthesized** distance notebook (`supporting/synthesized/distance_route/`) *is*
like-for-like with Figure 2 — same sign rule, same per-trial accounting — so that is the
pair to read against each other.

Keep [B1/12]'s two axes apart throughout: *noise tolerance for a fixed split* $\ne$ *which
split gets selected*. Only the former is in view here.

## Output

Figures are written **outside** `docs/overleafs/v9/figures/` on purpose — see the `FIG_DIR`
cell. Nothing in the manuscript, `paper_figures.py` or `PAPER_MAP.md` is touched.

## Cell 1 — Configuration

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "setup.py").exists():
    ROOT = ROOT.parent
PROJECT_ROOT = ROOT / "sub_sampled_fielder_vec"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analysis.utils.sweep_plots_two_panel import plot_pstar_pair, median_ratio_C
from analysis.utils.sweep_plots import compute_pstar
from src.cache_io import CACHE_ROOT, notebook_dir
from src.utils.eta_pool_griffing import collect_griffing_sweeps
from src.utils.eta_pool_sweep import (
    ETA_SAMPLE_CAP, P_VALUES, discover_ns_and_samples,
)

# --- toggles: copied from sec5_empirical/generated/eta_pool_sweep.ipynb ------------
ETA_TARGETS = [1, 5, 10, 15]
NS_INCLUDE  = [500, 1000, 2000, 4000, 6000, 8000]
METRIC      = "nmi"
METRIC_LABEL = "NMI"
AGG         = "mean"      # across pool samples
THRESHOLD   = 0.90        # p* = smallest grid p with mean NMI >= THRESHOLD. 0.90 not 0.95:
#                           on a 25-point log grid, 0.95 lands on the flat top of the
#                           transition where one noisy pool sample moves p* a whole step.
REPS        = 10          # sub-sampling reps per (sample, p) -- matches Fig 3's BOOTSTRAP_REPS
#                           in count, though not in how they are combined (see the header).

# ARPACK for the single leading eigenpair. The historical path is a full O(m^3) eigh to use
# one column: 14.9 s vs 0.12 s at n=6000, i.e. ~27 h vs ~50 min for n=6000 alone on this
# grid. Verified |cos| = 1 to 10 decimals and identical sign partitions against the dense
# path on these very pool matrices, full and sub-sampled.
EIGSOLVER = "lm_k1"

# Deliberately NOT docs/overleafs/v9/figures/ -- that tree is validated by
# scripts/sync_paper_figures.py --check, which errors ORPHAN on any PNG there lacking a
# paper_figures.py entry, and this notebook produces no paper figure (yet).
FIG_DIR = notebook_dir("supporting/distance_route")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"eta={ETA_TARGETS}  metric={METRIC}  agg={AGG}  threshold={THRESHOLD}")
print(f"|P_VALUES|={len(P_VALUES)} ({P_VALUES[0]:.0e}..{P_VALUES[-1]:.0f})  reps={REPS}")
print(f"eigsolver={EIGSOLVER}  sample cap per eta={ETA_SAMPLE_CAP}")
print(f"fig_dir={FIG_DIR}")

## Cell 2 — Discover the pool

Same pool, same per-$\eta$ sample caps as Figure 3, so the two figures average over the
same trees. `n=3000` exists on disk but is excluded by `NS_INCLUDE`, matching the paper
figure.

In [ ]:
ns_all, samples_by_n_eta = discover_ns_and_samples(CACHE_ROOT, eta_targets=ETA_TARGETS)
ns = [n for n in ns_all if n in NS_INCLUDE]
print(f"pool has n={ns_all}; using n={ns}")
print(f"  {'n':>6}  " + "  ".join(f"eta={e:>2}" for e in ETA_TARGETS))
for n in ns:
    counts = [len(samples_by_n_eta.get((n, e), [])) for e in ETA_TARGETS]
    print(f"  {n:>6}  " + "  ".join(f"{c:>6d}" for c in counts))
total = sum(len(samples_by_n_eta.get((n, e), [])) for n in ns for e in ETA_TARGETS)
print(f"\n{total} pool samples x {len(P_VALUES)} p x {REPS} reps "
      f"= {total * len(P_VALUES) * REPS:,} sub-sampled B-solves")

## Cell 3 — Sub-sampling sweep

Every `(n, eta, sample)` sweep is cached individually under `cache/bpart_sweep/`, so this
is resumable and safely interruptible; re-running after adding an `n` computes only the new
samples. Smallest `n` first, so the small panels are assemblable before the `n=8000` tail.

In [ ]:
_counts = {"computed": 0, "cached": 0}

def _on_sample(n, eta, idx, was_cached):
    _counts["cached" if was_cached else "computed"] += 1
    if not was_cached:
        print(f"    COMPUTE n={n:>5} eta={eta:>2} sample={idx:04d}", flush=True)

results_dist = collect_griffing_sweeps(
    ns, eta_targets=ETA_TARGETS, p_values=P_VALUES, reps=REPS,
    metric=METRIC, eigsolver=EIGSOLVER, on_sample=_on_sample,
)
print(f"sweep complete: computed={_counts['computed']} cached={_counts['cached']}")

print(f"\n  {'n':>6}  " + "  ".join(f"eta={e:>2}" for e in ETA_TARGETS))
for n in ns:
    print(f"  {n:>6}  " + "  ".join(
        f"{results_dist[(n, e)].shape[0]:>6d}" for e in ETA_TARGETS))
print("  (rows = pool samples contributing to each cell)")

# Shape + endpoint guard.
bad = []
for n in ns:
    for eta in ETA_TARGETS:
        arr = results_dist.get((n, eta))
        if arr is None or arr.shape[0] == 0:
            bad.append(f"(n={n}, eta={eta}) empty")
        elif arr.shape[1] != len(P_VALUES):
            bad.append(f"(n={n}, eta={eta}) shape {arr.shape}")
        elif arr[:, -1].mean() < 0.999:
            bad.append(f"(n={n}, eta={eta}) NMI at p=1 is {arr[:, -1].mean():.4f}")
assert not bad, "sweep output malformed: " + "; ".join(bad)
print(f"\nall {len(ns) * len(ETA_TARGETS)} cells non-empty, "
      f"width {len(P_VALUES)}, reaching NMI=1.0 at p=1")

## Cell 4 — One two-panel figure per $\eta$

`plot_pstar_pair` **unchanged** — the function that draws the paper's Figs 2 and 3 — so
panel geometry, type sizes, ticks, the dashed threshold line and the dotted $C\log n/n$
reference match the similarity figures exactly. Defaults are used for `n_ticks` and
`size_label` because this sweep spans the same $n=500\ldots8000$ as Figure 3.

In [ ]:
for eta in ETA_TARGETS:
    plot_pstar_pair(
        results_dist, P_VALUES, eta, ns,
        metric_label=METRIC_LABEL, agg=AGG, threshold=THRESHOLD,
        savepath=FIG_DIR / f"pstar_gen_dist_eta{eta}.png",
    )

## Cell 5 — Measured $C$

**Diagnostic scaffolding, not a paper metric.** $C$ is the median of the per-point ratios
$\hat p^\star_i/(\log n_i/n_i)$ (`median_ratio_C`) — the estimator the paper's constants are
quoted from, and *not* interchangeable with the least-squares `_theory_ref` behind the
appendix figures. `spread` is $\max_i r_i/\min_i r_i$: how badly a single constant fails to
describe the points.

No similarity column here, unlike the synthesized notebook: Figure 3's numbers come from a
different rounding rule and a different rep accounting (see the header), so putting them in
one table would invite exactly the comparison [B1/07] had struck from the manuscript. The
like-for-like $C$ comparison lives in `supporting/synthesized/distance_route/`.

In [ ]:
pstar_dist = compute_pstar(results_dist, P_VALUES, ETA_TARGETS, ns,
                           agg=AGG, threshold=THRESHOLD)

rows = []
for eta in ETA_TARGETS:
    pts = pstar_dist.get(eta, [])
    rec = {"eta": eta, "n_points": f"{len(pts)}/{len(ns)}"}
    if len(pts) >= 2:
        xs = np.array([n for n, _ in pts], float)
        ys = np.array([v for _, v in pts], float)
        C, spread = median_ratio_C(xs, ys)
        rec["C"] = round(C, 2)
        rec["spread"] = round(spread, 1)
    else:
        rec["C"] = rec["spread"] = np.nan
    rows.append(rec)

print("p* = C log n / n,  C = median of per-point ratios  (DIAGNOSTIC -- not a paper metric)")
print("n_points = sizes where mean NMI actually crosses the threshold\n")
pd.DataFrame(rows)[["eta", "C", "spread", "n_points"]]

In [ ]:
# The p-hat-star read-offs behind the table: a constant is only as meaningful as the number
# of points it summarizes.
detail = []
for eta in ETA_TARGETS:
    d = dict(pstar_dist.get(eta, []))
    for n in ns:
        detail.append({"eta": eta, "n": n, "pstar": d.get(n, np.nan)})
pd.DataFrame(detail).pivot(index="n", columns="eta", values="pstar").round(5)